In [1]:
# Pipeline Configuration
RUN_MODE = "production"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 4
CHUNK_SIZE = 500
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = False
PIPELINE_VERSION = "1.0.0"


# !pip -q install geopandas rasterio rioxarray pystac-client planetary-computer odc-stac shapely pyproj xarray folium leafmap




In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point

import rasterio
import rioxarray

import planetary_computer
import pystac_client




In [4]:
import os

folders = [
    "../data",
    "../data/raw",
    "../data/processed",
    "../data/features",
    "../data/metadata",
    "../data/final",
    "../data/lucas",
    "../data/sentinel",
    "../data/weather",
    "../data/soilgrids",
    "../outputs",
    "../outputs/csv",
    "../outputs/maps",
    "../outputs/figures",
    "../outputs/reports",
    "../outputs/metrics",
    "../outputs/learning_curves",
    "../outputs/feature_importance",
    "../outputs/confusion_matrix",
    "../models",
    "../models/machine_learning",
    "../models/deep_learning",
    "../models/ensemble",
    "../models/best",
    "../models/experimental"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")





Created: ../data
Created: ../data/raw
Created: ../data/processed
Created: ../data/features
Created: ../data/metadata
Created: ../data/final
Created: ../data/lucas
Created: ../data/sentinel
Created: ../data/weather
Created: ../data/soilgrids
Created: ../outputs
Created: ../outputs/csv
Created: ../outputs/maps
Created: ../outputs/figures
Created: ../outputs/reports
Created: ../outputs/metrics
Created: ../outputs/learning_curves
Created: ../outputs/feature_importance
Created: ../outputs/confusion_matrix
Created: ../models
Created: ../models/machine_learning
Created: ../models/deep_learning
Created: ../models/ensemble
Created: ../models/best
Created: ../models/experimental


In [5]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected Successfully")




Connected Successfully


# # collections = catalog.get_collections()

# # for c in collections:
    # # print(c.id)



print("Skipping collections listing for speed.")




from shapely.geometry import Point

lon = 31.083
lat = 30.563

point = Point(lon, lat)

buffer = point.buffer(0.003)

geometry = buffer.__geo_interface__

geometry




In [8]:
import os
import json
import time
import glob
import datetime
import gc
import pandas as pd
import numpy as np
from shapely.geometry import Point
import rioxarray
from odc.stac import stac_load

def process_point(row):
    point_id = int(row["POINT_ID"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    date_str = str(row["Survey_Date"])
    meta = {
        "POINT_ID": point_id, "Latitude": lat, "Longitude": lon, "Survey_Date": date_str,
        "Extraction_Time": datetime.datetime.now().isoformat(), "Pipeline_Version": PIPELINE_VERSION,
        "DEM_Source": "cop-dem-glo-30",
        "dem_ok": np.nan
    }
    patch_tif = f"../data/features/image_patches/dem/dem_{point_id}.tif"
    try:
        if ENABLE_CACHE and os.path.exists(patch_tif):
            rds = rioxarray.open_rasterio(patch_tif)
            z = rds[0].values
            meta_crs = str(rds.rio.crs)
        else:
            point = Point(lon, lat)
            buffer = point.buffer(0.003)
            geometry = buffer.__geo_interface__
            
            best_item = None
            last_err = None
            for retry in range(MAX_RETRIES + 1):
                try:
                    search = catalog.search(collections=["cop-dem-glo-30"], intersects=geometry)
                    items = list(search.items())
                    if not items:
                        raise ValueError(f"No DEM items found for {point_id}")
                    best_item = items[0]
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(2 ** retry)
            if best_item is None:
                raise last_err

            dem = None
            for retry in range(MAX_RETRIES + 1):
                try:
                    dem = stac_load(
                        [best_item], bands=["data"], geopolygon=geometry, crs="utm", resolution=30
                    )
                    break
                except Exception as e:
                    last_err = e
                    time.sleep(2 ** retry)
            if dem is None:
                raise last_err

            dem_squeezed = dem.isel(time=0)
            z = dem_squeezed["data"].values
            meta_crs = str(dem.rio.crs)
            # Save patch cache GeoTIFF
            stacked_da = dem_squeezed.to_array()
            stacked_da.rio.to_raster(patch_tif)
        dx = 30.0
        dy = 30.0
        try:
            dz_dy, dz_dx = np.gradient(z, dy, dx)
            slope = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
            aspect = np.degrees(np.arctan2(-dz_dx, dz_dy))
            aspect = (aspect + 360) % 360
        except Exception as e:
            slope = np.zeros_like(z)
            aspect = np.zeros_like(z)
        features = {
            "elevation": float(z.mean()),
            "slope": float(slope.mean()),
            "aspect": float(aspect.mean())
        }
        meta.update(features)
        meta["dem_ok"] = 1.0
        meta["CRS"] = meta_crs
        return {"status": "success", "data": meta}
    except Exception as e:
        err_msg = str(e)
        features = {
            "elevation": np.nan,
            "slope": np.nan,
            "aspect": np.nan
        }
        meta.update(features)
        meta["dem_ok"] = 0.0
        meta["CRS"] = np.nan
        return {"status": "success", "data": meta}


8


for i, item in enumerate(items):
    print("="*60)
    print("Item:", i)
    print("Date:", item.datetime)
    print("Cloud Cover:", item.properties["eo:cloud_cover"])
    print("ID:", item.id)




best_item = sorted(
    items,
    key=lambda x: x.properties["eo:cloud_cover"]
)[0]

print(best_item.datetime)
print(best_item.properties["eo:cloud_cover"])




from odc.stac import stac_load




dataset = stac_load(
    [best_item],
    bands=["B02","B03","B04","B08","B11","B12"],
    geopolygon=geometry,
    resolution=10,
    chunks={}
)




dataset




# تحويل كل الباندات إلى Float Reflectance

dataset = dataset.astype("float32") / 10000

dataset




nir = dataset.B08
red = dataset.B04

ndvi = (nir - red) / (nir + red)

ndvi




print("Minimum :", float(ndvi.min()))
print("Maximum :", float(ndvi.max()))
print("Mean    :", float(ndvi.mean()))




import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

ndvi.squeeze().plot(
    cmap="RdYlGn",
    vmin=-1,
    vmax=1
)

plt.title("NDVI")
plt.axis("off")
plt.show()




blue = dataset.B02.squeeze()
green = dataset.B03.squeeze()
red = dataset.B04.squeeze()
nir = dataset.B08.squeeze()
swir1 = dataset.B11.squeeze()
swir2 = dataset.B12.squeeze()




import numpy as np

# Vegetation
NDVI = (nir - red) / (nir + red)

EVI = 2.5 * (
    (nir - red) /
    (nir + 6 * red - 7.5 * blue + 1)
)

SAVI = 1.5 * (
    (nir - red) /
    (nir + red + 0.5)
)

MSAVI = (
    (2 * nir + 1)
    - np.sqrt((2 * nir + 1) ** 2 - 8 * (nir - red))
) / 2

GNDVI = (nir - green) / (nir + green)

NDMI = (nir - swir1) / (nir + swir1)

NDWI = (green - nir) / (green + nir)

BSI = (
    (swir1 + red) - (nir + blue)
) / (
    (swir1 + red) + (nir + blue)
)




dataset = stac_load(
    [best_item],
    bands=["B02","B03","B04","B05","B08","B11","B12"],
    geopolygon=geometry,
    resolution=10,
    chunks={}
)

dataset = dataset.astype("float32") / 10000




rededge = dataset.B05.squeeze()

NDRE = (nir - rededge) / (nir + rededge)

CIre = (nir / rededge) - 1




Brightness = np.sqrt((red**2 + nir**2) / 2)

SoilColor = red / green

ClayIndex = swir1 / swir2




indices = {
    "NDVI": NDVI,
    "EVI": EVI,
    "SAVI": SAVI,
    "MSAVI": MSAVI,
    "GNDVI": GNDVI,
    "NDMI": NDMI,
    "NDWI": NDWI,
    "BSI": BSI,
    "NDRE": NDRE,
    "CIre": CIre,
    "Brightness": Brightness,
    "SoilColor": SoilColor,
    "ClayIndex": ClayIndex
}

for name, img in indices.items():
    print(
        name,
        "Min:", float(img.min()),
        "Max:", float(img.max()),
        "Mean:", float(img.mean())
    )




import matplotlib.pyplot as plt

for name, img in indices.items():

    plt.figure(figsize=(6,6))

    img.plot(cmap="RdYlGn")

    plt.title(name)

    plt.axis("off")

    plt.show()




import numpy as np
import pandas as pd

def extract_s2_features(dataset):

    blue  = dataset.B02.squeeze()
    green = dataset.B03.squeeze()
    red   = dataset.B04.squeeze()

    rededge = dataset.B05.squeeze()

    nir = dataset.B08.squeeze()

    swir1 = dataset.B11.squeeze()
    swir2 = dataset.B12.squeeze()

    features = {}

    features["NDVI"] = float(((nir-red)/(nir+red)).mean())

    features["EVI"] = float((
        2.5*((nir-red)/(nir+6*red-7.5*blue+1))
    ).mean())

    features["SAVI"] = float((
        1.5*((nir-red)/(nir+red+0.5))
    ).mean())

    features["MSAVI"] = float(((
        (2*nir+1) -
        np.sqrt((2*nir+1)**2-8*(nir-red))
    )/2).mean())

    features["GNDVI"] = float(((nir-green)/(nir+green)).mean())

    features["NDMI"] = float(((nir-swir1)/(nir+swir1)).mean())

    features["NDWI"] = float(((green-nir)/(green+nir)).mean())

    features["BSI"] = float(((
        (swir1+red)-(nir+blue)
    )/(
        (swir1+red)+(nir+blue)
    )).mean())

    features["NDRE"] = float(((nir-rededge)/(nir+rededge)).mean())

    features["CIre"] = float(((nir/rededge)-1).mean())

    features["Brightness"] = float(
        np.sqrt((red**2+nir**2)/2).mean()
    )

    features["ClayIndex"] = float(
        (swir1/swir2).mean()
    )

    return pd.DataFrame([features])




features = extract_s2_features(dataset)

features




# Exploratory CSV write commented out for multi-point pipeline execution


## ============================================================= ##

# !pip install pystac-client planetary-computer odc-stac rasterio rioxarray




import planetary_computer
import pystac_client
import odc.stac




catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)




geometry




# search = catalog.search(
#     collections=["sentinel-1-grd"],
#     intersects=geometry,
#     datetime="2024-05-01/2024-06-01"
# )

# items = list(search.items())

# print(len(items))



print("Skipping exploratory STAC search for speed and stability.")



item = items[0]

print(item.datetime)




data = odc.stac.load(
    [item],
    bands=["vv","vh"],
    geopolygon=geometry,
    resolution=10,
    crs="EPSG:4326"
)




data




vv = data["vv"].isel(time=0)
vh = data["vh"].isel(time=0)

print(vv)
print(vh)




print(data)




print(vv.shape)
print(vh.shape)




import numpy as np

print(np.isnan(vv.values).sum())
print(vv.values.size)




vv.values[:5, :5]




crs="EPSG:4326",
resolution=10




# vv_mean = float(vv.mean())
# vh_mean = float(vh.mean())

# print(vv_mean)
# print(vh_mean)




# rvi = 4 * vh / (vv + vh)




# rvi_mean = float(rvi.mean())

# print(rvi_mean)




data = odc.stac.load(
    [item],
    bands=["vv", "vh"],
    geopolygon=geometry,
    crs="utm",
    resolution=10
)




# data = odc.stac.load(
#     [item],
#     bands=["vv", "vh"],
#     geopolygon=geometry
# )




print(item.id)
print(item.datetime)

print(vv.shape)
print(vv.rio.crs)

print(np.nanmin(vv.values))
print(np.nanmax(vv.values))




print(data)




import numpy as np

print(np.isnan(vv.values).sum())
print(vv.values.size)




print(vv.values.min())
print(vv.values.max())




vv.mean()




np.nanmean(vv.values)




np.nanmean(vh.values)




vv.mean(skipna=True)
vh.mean(skipna=True)




vv = data["vv"].isel(time=0)
vh = data["vh"].isel(time=0)

print(vv.shape)
print(vh.shape)




import numpy as np

print(np.nanmean(vv.values))
print(np.nanmean(vh.values))




vv = data["vv"].isel(time=0)
vh = data["vh"].isel(time=0)

print(vv.shape)

import numpy as np
print(np.isnan(vv.values).sum())
print(vv.values.size)
print(np.nanmean(vv.values))




import numpy as np

vv_linear = np.nanmean(vv.values)
vh_linear = np.nanmean(vh.values)

print(vv_linear)
print(vh_linear)




vv_db = 10 * np.log10(vv_linear)
vh_db = 10 * np.log10(vh_linear)

print(vv_db)
print(vh_db)




import numpy as np

rvi = (4 * vh) / (vv + vh)

rvi_mean = float(np.nanmean(rvi.values))

print(rvi_mean)




# search = catalog.search(
#     collections=["cop-dem-glo-30"],
#     intersects=geometry
# )

# dem_items = list(search.items())

# print(len(dem_items))



print("Skipping exploratory STAC search for speed and stability.")



dem_item = dem_items[0]

print(dem_item.collection_id)
print(dem_item.id)




print(dem_item.assets.keys())




dem = odc.stac.load(
    [dem_item],
    bands=["data"],
    geopolygon=geometry,
    resolution=30
)




elevation = dem["data"].isel(time=0)

import numpy as np

print(np.nanmean(elevation.values))




print(dem_item.collection_id)
print(dem_item.assets.keys())
print(np.nanmean(elevation.values))




print(dem)




print(dem["data"].shape)




print(dem["data"].values)




resolution=30





dem = odc.stac.load(
    [dem_item],
    bands=["data"],
    like=data
)




elevation = dem["data"].isel(time=0)

import numpy as np

print(np.nanmean(elevation.values))




elevation = dem["data"].isel(time=0)

import numpy as np

print(np.nanmean(elevation.values))




print(dem["data"].shape)




import numpy as np

arr = dem["data"].values

print("Shape:", arr.shape)
print("Total Pixels:", arr.size)
print("NaN Count:", np.isnan(arr).sum())
print("Finite Count:", np.isfinite(arr).sum())




print(arr[0, :5, :5])




import numpy as np
import matplotlib.pyplot as plt

# Get the 2D elevation array (assuming the time dimension was squeezed previously or we take the first index)
elevation_2d = elevation.values
pixel_size = 10 # meters, based on the `resolution` used when loading the data

# Calculate gradients in x and y directions
dy, dx = np.gradient(elevation_2d, pixel_size)

# Calculate slope in degrees
slope_radians = np.arctan(np.sqrt(dx**2 + dy**2))
slope_degrees = np.degrees(slope_radians)

# Calculate aspect in degrees (direction of the steepest slope)
aspect_radians = np.arctan2(dy, -dx)
aspect_degrees = (np.degrees(aspect_radians) + 360) % 360 # Convert to 0-360 range

print("Mean Slope :", np.nanmean(slope_degrees))
print("Mean Aspect:", np.nanmean(aspect_degrees))




### Visualizing Slope and Aspect

plt.figure(figsize=(8,8))
plt.imshow(slope_degrees, cmap="terrain")
plt.colorbar(label="Slope (degrees)")
plt.title("Calculated Slope")
plt.show()




plt.figure(figsize=(8,8))
plt.imshow(aspect_degrees, cmap="hsv")
plt.colorbar(label="Aspect (degrees)")
plt.title("Calculated Aspect")
plt.show()




print(dem.coords)
print(dem.rio.crs)




from shapely.geometry import shape

poly = shape(geometry)

print(poly.bounds)




print(dem)




print(dem["data"])




import numpy as np

elevation = dem["data"].isel(time=0).values

x = dem["x"].values
y = dem["y"].values

dx = np.mean(np.diff(x))
dy = np.mean(np.diff(y))

print(dx)
print(dy)




import numpy as np

# DEM
z = dem["data"].isel(time=0).values

# Pixel size
dx = abs(np.mean(np.diff(dem.x.values)))
dy = abs(np.mean(np.diff(dem.y.values)))

# Gradient
dz_dy, dz_dx = np.gradient(z, dy, dx)

# Slope (degrees)
slope = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

# Aspect (degrees)
aspect = np.degrees(np.arctan2(-dz_dx, dz_dy))
aspect = (aspect + 360) % 360

print("Elevation Mean :", np.mean(z))
print("Elevation Min  :", np.min(z))
print("Elevation Max  :", np.max(z))

print("Slope Mean     :", np.mean(slope))
print("Slope Max      :", np.max(slope))

print("Aspect Mean    :", np.mean(aspect))




In [86]:
import os
import json
import time
import glob
import datetime
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
import rasterio
import rioxarray
from pyproj import Transformer
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import odc.stac

folders = ["../data/features/image_patches/dem", "../data/metadata", "../logs"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

df_lucas = pd.read_csv("../data/features/lucas_labels.csv")
checkpoint_file = "../data/metadata/dem_checkpoint.json"
chunk_dir = "../data/features/dem_chunks"
os.makedirs(chunk_dir, exist_ok=True)
failed_points_file = "../data/metadata/failed_points.csv"
log_file = "../logs/dem.log"

def log_msg(msg):
    timestamp = datetime.datetime.now().isoformat()
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {msg}\n")
    print(msg)

start_idx = 0
if ENABLE_CHECKPOINT and os.path.exists(checkpoint_file):
    try:
        with open(checkpoint_file, "r") as f:
            cp = json.load(f)
            start_idx = cp.get("last_processed_idx", -1) + 1
        log_msg(f"Resuming DEM from index {start_idx}")
    except Exception as e:
        log_msg(f"Failed to read checkpoint: {e}")

def process_point(row):
    point_id = int(row["POINT_ID"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    date_str = str(row["Survey_Date"])
    meta = {
        "POINT_ID": point_id, "Latitude": lat, "Longitude": lon, "Survey_Date": date_str,
        "Extraction_Time": datetime.datetime.now().isoformat(), "Pipeline_Version": PIPELINE_VERSION,
        "DEM_Source": "cop-dem-glo-30",
        "dem_ok": np.nan
    }
    patch_tif = f"../data/features/image_patches/dem/dem_{point_id}.tif"
    try:
        if ENABLE_CACHE and os.path.exists(patch_tif):
            rds = rioxarray.open_rasterio(patch_tif)
            z = rds[0].values
            meta_crs = str(rds.rio.crs)
        else:
            point = Point(lon, lat)
            buffer = point.buffer(0.003)
            geometry = buffer.__geo_interface__
            search = catalog.search(collections=["cop-dem-glo-30"], intersects=geometry)
            items = list(search.items())
            if not items:
                raise ValueError(f"No DEM items found for {point_id}")
            best_item = items[0]
            dem = odc.stac.load(
                [best_item], bands=["data"], geopolygon=geometry, crs="utm", resolution=30
            )
            dem_squeezed = dem.isel(time=0)
            z = dem_squeezed["data"].values
            meta_crs = str(dem.rio.crs)
            # Save patch cache GeoTIFF
            stacked_da = dem_squeezed.to_array()
            stacked_da.rio.to_raster(patch_tif)
        dx = 30.0
        dy = 30.0
        try:
            dz_dy, dz_dx = np.gradient(z, dy, dx)
            slope = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
            aspect = np.degrees(np.arctan2(-dz_dx, dz_dy))
            aspect = (aspect + 360) % 360
        except Exception as e:
            slope = np.zeros_like(z)
            aspect = np.zeros_like(z)
        features = {
            "elevation": float(z.mean()),
            "slope": float(slope.mean()),
            "aspect": float(aspect.mean())
        }
        meta.update(features)
        meta["dem_ok"] = 1.0
        meta["CRS"] = meta_crs
        if SAVE_PREVIEW_IMAGES:
            patch_png = f"../data/features/image_patches/dem/dem_{point_id}.png"
            plt.imsave(patch_png, z, cmap="terrain")
        return {"status": "success", "data": meta}
    except Exception as e:
        err_msg = str(e)
        failed_row = {
            "POINT_ID": point_id, "longitude": lon, "latitude": lat, "date": date_str,
            "error_message": err_msg, "timestamp": datetime.datetime.now().isoformat()
        }
        pd.DataFrame([failed_row]).to_csv(
            failed_points_file, mode="a", header=not os.path.exists(failed_points_file), index=False
        )
        return {"status": "failed", "point_id": point_id, "error": err_msg}

def run_proc(row_dict):
    return process_point(row_dict)

num_samples = len(df_lucas)
start_time = time.time()
for c_idx in range(start_idx, num_samples, CHUNK_SIZE):
    chunk = df_lucas.iloc[c_idx : min(c_idx + CHUNK_SIZE, num_samples)]
    chunk_data = []
    log_msg(f"DEM Processing batch {c_idx} to {min(c_idx+CHUNK_SIZE, num_samples)}...")
    try:
        rows_list = [row.to_dict() for _, row in chunk.iterrows()]
        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(run_proc, r): r for r in rows_list}
            for fut in as_completed(futures):
                res = fut.result()
                if res["status"] == "success":
                    chunk_data.append(res["data"])
    except Exception as e:
        log_msg(f"ProcessPool failed: {e}. Falling back to ThreadPoolExecutor...")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_point, row): row for _, row in chunk.iterrows()}
            for fut in as_completed(futures):
                res = fut.result()
                if res["status"] == "success":
                    chunk_data.append(res["data"])
    if chunk_data:
        df_chunk = pd.DataFrame(chunk_data).drop_duplicates(subset=["POINT_ID"])
        df_chunk.to_csv(f"{chunk_dir}/dem_part_{c_idx//CHUNK_SIZE + 1:03d}.csv", index=False)
    if ENABLE_CHECKPOINT:
        with open(checkpoint_file, "w") as f:
            json.dump({"last_processed_idx": min(c_idx + CHUNK_SIZE - 1, num_samples - 1)}, f)
    elapsed = time.time() - start_time
    processed = min(c_idx + CHUNK_SIZE, num_samples) - start_idx
    remaining = num_samples - min(c_idx + CHUNK_SIZE, num_samples)
    eta = (elapsed / processed) * remaining if processed > 0 else 0
    log_msg(f"Progress DEM: {min(c_idx+CHUNK_SIZE, num_samples)}/{num_samples} ({processed/num_samples*100:.1f}%) | ETA: {eta/60:.1f} min")
    del chunk_data
    gc.collect()

all_chunks = sorted(glob.glob(f"{chunk_dir}/dem_part_*.csv" ))
if all_chunks:
    df_final = pd.concat([pd.read_csv(ch) for ch in all_chunks], ignore_index=True)
    df_final.to_csv("../data/features/dem_features.csv", index=False)
    log_msg(f"DEM Merge Complete. Total rows: {len(df_final)}")





DEM Processing batch 0 to 10...


ProcessPool failed: A process in the process pool was terminated abruptly while the future was running or pending.. Falling back to ThreadPoolExecutor...


Progress DEM: 10/10 (100.0%) | ETA: 0.0 min
DEM Merge Complete. Total rows: 10
